# Step 3 — the OSM segmentation subset (optional)

What `code/03_osm_segmentation.py` produced: the chips where OpenStreetMap has
actually mapped mining ground, with those polygons rasterized onto the chip's
own pixel grid as a mask.

    python code/03_osm_segmentation.py --stage query
    python code/03_osm_segmentation.py --stage masks
    python code/03_osm_segmentation.py --stage index

One caveat worth keeping in mind throughout: a chip without a mask does not
mean there is no pit, only that nobody mapped it. The masks are positive-only.

In [ ]:
import json, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "code"))
import common as C

# One fixed categorical order, used the same way in every chart and never
# cycled; a single hue carries magnitude. Validated for colour-vision
# deficiency, so series stay distinguishable.
CAT = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
SEQ = "Blues"
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#898781", "#e1e0d9", "#fcfcfb"
plt.rcParams.update({
    "figure.dpi": 110, "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "axes.edgecolor": GRID, "axes.labelcolor": INK,
    "text.color": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 10, "figure.figsize": (8, 4),
})

def report(name):
    """Load data/reports/<name>.json, or say which step writes it."""
    p = C.REPORTS / f"{name}.json"
    if not p.exists():
        raise SystemExit(f"{p.relative_to(REPO)} is missing — run the step that writes it.")
    return json.loads(p.read_text())

def need(path):
    p = Path(path)
    if not p.exists():
        raise SystemExit(f"{p} is missing — run the step that writes it first.")
    return p

def labelled_barh(ax, labels, values, color=CAT[0], fmt="{:,.0f}"):
    """Horizontal bars with the value written at the end of each one."""
    y = np.arange(len(labels))
    ax.barh(y, values, color=color, height=0.72)
    ax.set_yticks(y, labels)
    ax.invert_yaxis()
    ax.xaxis.set_visible(False)
    ax.grid(False)
    span = max(values) if len(values) else 1
    for yi, v in zip(y, values):
        ax.text(v + span * 0.01, yi, fmt.format(v), va="center", fontsize=9, color=MUTED)
    for s in ("left", "bottom"):
        ax.spines[s].set_visible(False)
    return ax


In [ ]:
seg_rep = report("03_seg")
seg = pd.read_csv(need(C.SEG_INDEX_CSV), low_memory=False)
print(f"{seg_rep['chips_with_mask']:,} of {seg_rep['chips_in']:,} chips "
      f"({100 * seg_rep['share']:.1f}%) have a mask")
print(f"mean mask covers {seg_rep['mask_frac']['mean']:.1%} of a chip")
seg.head(3)

## How much of a chip the mask covers

A tiny mask fraction usually means an OSM node (a shaft) buffered to a pixel rather than a mapped pit outline; the large end is open-pit mines that fill the window.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.4))
axes[0].hist(100 * seg.mask_frac, bins=np.linspace(0, 100, 41), color=CAT[0])
axes[0].set(xlabel="chip pixels inside a mining polygon (%)", ylabel="chips",
            yscale="log")
axes[1].hist(seg.n_features.clip(upper=20), bins=np.arange(0.5, 22), color=CAT[1])
axes[1].set(xlabel="OSM features in the chip (clipped at 20)", ylabel="chips")
plt.tight_layout()

## The masks on the imagery

Nine of the largest masks, outlined over the SWIR composite. This is the check that the transform stored in each chip really lines the mask up with the pixels.

In [ ]:
import matplotlib.patheffects as pe

pick = seg.head(9)
fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for ax, r in zip(axes.ravel(), pick.itertuples()):
    chip = C.DATA / r.chip_path
    m = np.load(C.DATA / r.mask_path, allow_pickle=False)["mask"]
    d = np.load(chip, allow_pickle=False)
    wl, X, valid = d["wavelength"], d["X"], d["valid"]
    refl = np.where(valid[None] == 1, X * float(d["scale_factor"]), np.nan)
    at = lambda nm: int(np.argmin(np.abs(wl - nm)))
    rgb = np.dstack([refl[at(nm)] for nm in (2200, 1650, 1050)])
    lo, hi = np.nanpercentile(rgb, [2, 98])
    ax.imshow(np.nan_to_num(np.clip((rgb - lo) / max(1e-9, hi - lo), 0, 1)))
    ax.contour(m, levels=[0.5], colors=[CAT[1]], linewidths=1.6)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    t = ax.set_title(f"{r.site_id} · {r.mask_frac:.0%}", fontsize=8)
    t.set_path_effects([pe.withStroke(linewidth=2, foreground=SURFACE)])
for ax in axes.ravel()[len(pick):]:
    ax.axis("off")
plt.tight_layout()

## Where the segmentation subset is

Smaller and more clustered than the classification set, because OSM mining coverage is uneven by country.

In [ ]:
idx = pd.read_csv(need(C.CHIP_INDEX_CSV), low_memory=False)
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(idx.lon, idx.lat, s=0.6, c=GRID, linewidths=0, label=f"all chips ({len(idx):,})")
ax.scatter(seg.lon, seg.lat, s=3, c=CAT[1], linewidths=0,
           label=f"with an OSM mask ({len(seg):,})")
ax.set(xlim=(-180, 180), ylim=(-90, 90), xlabel="longitude", ylabel="latitude")
ax.set_aspect("equal")
ax.legend(loc="lower left", frameon=False, fontsize=9)
plt.tight_layout()